## 📦 Environment Setup & Imports

In [1]:
import os
import json
import sys
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

project_root = Path.cwd().parent
sys.path.append(str(project_root / "src"))

load_dotenv(project_root / ".env")
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

from rag_pipeline import (
    load_data,
    build_or_load_faiss_index,
    check_polypharmacy as check_polypharmacy_orig,
    check_polypharmacy_light as check_polypharmacy_light_orig,
)

kb_df, lookups = load_data()
index, embeddings = build_or_load_faiss_index(kb_df, client)


def check_polypharmacy(drug_query: str):
    text = drug_query.replace("interaction", "").replace("and", ",")
    drugs = [d.strip() for d in text.split(",") if d.strip()]
    return check_polypharmacy_orig(drugs, kb_df, lookups, index, embeddings, client)


def check_polypharmacy_light(drug_query: str):
    text = drug_query.replace("interaction", "").replace("and", ",")
    drugs = [d.strip() for d in text.split(",") if d.strip()]
    return check_polypharmacy_light_orig(drugs, kb_df, lookups)


print("Environment initialized successfully")

Loading knowledge base...
   Loaded 170,782 interactions
   Unique drug pairs: 170,782

Loading RxNorm mappings...
   Name to RxCUI: 157,972
   Brand to Ingredient: 77,518
Loading existing FAISS index...
   Loaded index with 170,782 vectors (3072 dims)
Environment initialized successfully


## 🎯 Severity Classification (GPT-5)

In [2]:
def classify_severity_gpt5(interaction_text):
    """
    Use GPT-5 to classify drug interaction severity.

    Args:
        interaction_text: Single interaction description from KB

    Returns:
        dict with severity (RED/YELLOW/GREEN) and brief clinical reasoning
    """
    prompt = f"""You are a medical doctor and clinical pharmacist reviewing a drug-drug interaction.

Interaction evidence:
{interaction_text}

Task:
Determine the CLINICAL SEVERITY of this interaction even if words such as
"contraindicated", "major", "moderate", or "minor" do NOT appear.
Use both the provided evidence and your own pharmacologic knowledge.

Classification rules:
- 🟥 (Contraindicated): Life-threatening or severe interaction - avoid combination entirely.
- 🟨 (Caution): Clinically significant or moderate risk - requires monitoring or dose adjustment.
- 🟩 (No Interaction): No meaningful pharmacologic or clinical interaction expected.

Return ONLY valid JSON - no markdown, no commentary, no code fences.
Output must start with {{ and end with }}.

Expected format:
{{"severity": "🟥" or "🟨" or "🟩", "explanation": "brief reasoning"}}

Examples:
- "may increase bleeding risk" → 🟨
- "contraindicated with..." → 🟥
- "no interaction known" or "minimal clinical effect" → 🟩
"""
    try:
        response = client.responses.create(
            model="gpt-5",
            input=prompt,
            text={"format": {"type": "text"}}
        )

        result_text = (response.output_text or "").strip()

        if not result_text or not result_text.startswith("{") or not result_text.endswith("}"):
            raise ValueError("Invalid JSON format from GPT-5")

        return json.loads(result_text)

    except (json.JSONDecodeError, Exception) as e:
        raise


print("Severity classifier defined")

Severity classifier defined


In [3]:
# Test the classifier
test_interaction = "The metabolism of Diphenhydramine can be decreased when combined with Acetaminophen."
result = classify_severity_gpt5(test_interaction)
print(f"   Evidence: {test_interaction}")
print(f"   Severity: {result['severity']}")
print(f"   Reasoning: {result['explanation']}")

   Evidence: The metabolism of Diphenhydramine can be decreased when combined with Acetaminophen.
   Severity: 🟩
   Reasoning: Acetaminophen is not a meaningful inhibitor of CYP2D6 (primary pathway for diphenhydramine). Any reduction in diphenhydramine metabolism is minimal and not clinically significant; the combination is widely co-formulated without dose adjustment.
